In [ ]:
from _init import *

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [ ]:
import random, torch, re

from transformers import PreTrainedTokenizerFast, AutoModelForCausalLM

from bait.utils import common_utils, json_utils, container_utils, file_utils, model_utils, tokenizer_utils, context_utils
from bait.utils.common_utils import safe_division
from bait.utils.container_utils import chunks, add_str_num
from bait.core.bait_prompts import FILE_FORMATS, CONTEXT_SIZE, get_generate_prompt

In [ ]:
SEED = 42
common_utils.set_seed(SEED)

In [ ]:
work_dir = f'/home/nlpshlee/dev_env/git/repos/bait'
data_dir = f'{work_dir}/data'
in_dir = f'{data_dir}/create_contexts'
out_dir = f'{data_dir}/check_confirmation_bias'

dtype = 'bfloat16'
device = 'cuda:0'
max_seq_length = 4096
max_new_tokens = 64

In [ ]:
def mix_contexts(contexts_fact_dict: dict, contexts_counter_dict: dict, ext_n_fact: int, ext_n_counter: int):
    if ext_n_fact <= len(contexts_fact_dict) and ext_n_counter <= len(contexts_counter_dict):
        ext_contexts_fact = random.sample(list(contexts_fact_dict.values()), ext_n_fact)
        ext_contexts_counter = random.sample(list(contexts_counter_dict.values()), ext_n_counter)

        # 셔플 전 각각의 컨텍스트에 태그(출처)를 붙여 튜플 형태로 결합
        tagged_contexts = [(ctx, 'fact') for ctx in ext_contexts_fact] + [(ctx, 'counter') for ctx in ext_contexts_counter]

        # 태그를 붙인 상태에서 셔플
        random.shuffle(tagged_contexts)

        # 태그를 제거하고 위치 기록
        mixed_contexts = []
        fact_idxs, counter_idxs = [], []

        for i, (ctx, tag) in enumerate(tagged_contexts):
            mixed_contexts.append(ctx)
            if tag == 'fact':
                fact_idxs.append(i)
            else:
                counter_idxs.append(i)

        return mixed_contexts, fact_idxs, counter_idxs

    return None, None, None

In [ ]:
# contexts_fact_dict = {
#     '1': 'fact context 1',
#     '2': 'fact context 2',
#     '3': 'fact context 3',
#     '4': 'fact context 4'
# }

# contexts_counter_dict = {
#     '4': 'counter context 4',
#     '5': 'counter context 5',
#     '6': 'counter context 6',
#     '7': 'counter context 7'
# }

# mixed_contexts, fact_idxs, counter_idxs = mix_contexts(contexts_fact_dict, contexts_counter_dict, 2, 3)

# print(f'mixed_contexts : {mixed_contexts}\n')
# print(f'fact_idxs : {fact_idxs}\n')
# print(f'counter_idxs : {counter_idxs}')

In [ ]:
def add_prompts(question: str, answer: str, contexts_fact: dict, contexts_counter: dict,
                prompts: list, answers: list, states: list,
                mixed_contexts_list: list, fact_idxs_list: list, counter_idxs_list: list):

    for file_format in FILE_FORMATS:
        contexts_fact_dict = contexts_fact[file_format]
        contexts_counter_dict = contexts_counter[file_format]

        for i in range(CONTEXT_SIZE):
            mixed_contexts, fact_idxs, counter_idxs = mix_contexts(contexts_fact_dict, contexts_counter_dict, i, CONTEXT_SIZE-1-i)

            if mixed_contexts is None:
                prompts.append(get_generate_prompt(question))
                states.append(False)
            else:
                prompts.append(get_generate_prompt(question, mixed_contexts))
                states.append(True)

            answers.append(answer)
            mixed_contexts_list.append(mixed_contexts)
            fact_idxs_list.append(fact_idxs)
            counter_idxs_list.append(counter_idxs)

In [ ]:
'''
    offset_mapping shape : torch.Size([922, 2])
    target_tok_attn shape : torch.Size([922]) / (현재는 마지막 토큰의 마지막 레이어에서의 어텐션)
'''
def make_result(prompt: str, contexts: list, offset_mapping, target_tok_attn: torch.Tensor):
    context_tok_idxs = context_utils.extract_context_tok_idxs(prompt, contexts, offset_mapping)

    # 각 컨텍스트 별로 어텐션 계산 (타겟 토큰 입장에서 컨텍스트에 속한 모든 토큰의 어텐션 합계)
    # 입력의 마지막 토큰이 앞에 컨텍스트 별로 얼마나 보고 있는지 계산
    context_attns = {}

    for context_idx, tok_idxs in context_tok_idxs.items():
        context_attns[context_idx] = target_tok_attn[tok_idxs].sum().item() if tok_idxs else 0.0

    # 타겟 토큰(마지막 토큰) 기준에서 프롬프트 전체에 대한 어텐션 합계
    prompt_attn = target_tok_attn.sum().item()

    # 타겟 토큰(마지막 토큰) 기준에서 모든 컨텍스트에 대한 어텐션 합계
    context_attn_total = sum(context_attns.values())

    return {
        'prompt': prompt,
        'contexts': contexts,
        'context_tok_cnts': {context_idx: len(tok_idxs) for context_idx, tok_idxs in context_tok_idxs.items()},
        'context_attns': context_attns,
        'context_percents_of_prompt': {context_idx: safe_division(context_attn, prompt_attn) for context_idx, context_attn in context_attns.items()},
        'context_percents_of_context_total': {context_idx: safe_division(context_attn, context_attn_total) for context_idx, context_attn in context_attns.items()},
        'prompt_attn': prompt_attn,
        'context_attn_total' : context_attn_total
    }

In [ ]:
def compute_attention_batch(model: AutoModelForCausalLM, tokenizer: PreTrainedTokenizerFast,
                            prompts: list, mixed_contexts_list: list, layers=[-1], batch_size=1,
                            reduce_heads='mean', query_token='last'):

    if len(prompts) == len(mixed_contexts_list):
        results = []

        for prompts_batch, mixed_contexts_batch in zip(chunks(prompts, batch_size), chunks(mixed_contexts_list, batch_size)):
            chat_prompts_batch, inputs = model_utils.make_inputs(tokenizer, device, prompts_batch, max_seq_length,
                                                           return_offsets_mapping=True, return_all=True)

            offset_mappings = inputs.pop('offset_mapping').cpu() # torch.Size([50, 923, 2]) == (Batch, Seq_len, 2)

            # 모델 Forward Pass 및 Attention 추출
            outputs = model(**inputs, output_attentions=True)
            num_layers = len(outputs.attentions)

            layer_idxs = list(range(num_layers)) if layers=='all' else [(l if l >= 0 else num_layers+l) for l in layers]
            # print(f'num_layers : {num_layers}, layer_idxs : {layer_idxs}')

            '''
                (타겟) 레이어 별로 마지막 토큰의 어텐션(헤드 평균) 저장
            '''
            layer_target_tok_attns = {}

            for layer_idx in layer_idxs:
                layer_attn = outputs.attentions[layer_idx] # torch.Size([1, 24, 922, 922]) == (Batch, Heads, Seq_len, Seq_len)

                if reduce_heads == 'mean':
                    layer_attn = layer_attn.mean(dim=1) # 멀티 헤드 평균 -> torch.Size([1, 922, 922]) == (Batch, Seq_len, Seq_len)

                    if query_token == 'last':
                        # left padding이 적용되었으므로 -1 위치는 각 시퀀스의 마지막 토큰(첫 생성 시점)
                        layer_last_tok_attn = layer_attn[:, -1, :] # torch.Size([1, 922]) == (Batch, Seq_len)

                        layer_target_tok_attns[f'{layer_idx}'] = layer_last_tok_attn

            # 타겟 토큰에 대하여 타겟 레이어의 평균 어텐션 / torch.Size([1, 922]) == (Batch, Seq_len)
            layer_target_tok_attns['avg'] = torch.stack(list(layer_target_tok_attns.values()), dim=0).mean(dim=0).float().cpu()

            del outputs
            common_utils.clear_gpu_memory()

            # 배치 단위
            for i in range(len(chat_prompts_batch)):
                result = make_result(
                    chat_prompts_batch[i], mixed_contexts_batch[i], offset_mappings[i], layer_target_tok_attns['avg'][i]
                )

                result['target_layers'] = layer_idxs
                results.append(result)

        return results

    return None

In [ ]:
def check_confirmation_bias(model: AutoModelForCausalLM, tokenizer: PreTrainedTokenizerFast, datas, batch_size1=1, batch_size2=1):
    data_size = len(datas)
    percents = {}

    for i, datas_batch in enumerate(container_utils.chunks(datas, batch_size1)):
        # 성능 측정을 위한 리스트
        prompts_batch, answers_batch, states_batch = [], [], []

        # 어텐션 측정을 위한 추가 리스트
        mixed_contexts_batch, fact_idxs_batch, counter_idxs_batch = [], [], []

        for data in datas_batch:
            question = data['question']
            answer_fact = data['answer_fact']
            answer_counter = data['answer_counter']
            contexts_fact = data['contexts_fact']
            contexts_counter = data['contexts_counter']
            
            add_prompts(question, answer_fact, contexts_fact, contexts_counter,
                        prompts_batch, answers_batch, states_batch,
                        mixed_contexts_batch, fact_idxs_batch, counter_idxs_batch)

        attn_results_batch = compute_attention_batch(model, tokenizer, prompts_batch, mixed_contexts_batch, [-1], batch_size2)

        # print(f'prompts_batch[0] : {prompts_batch[0]}\n')
        # print(f'answers_batch[0] : {answers_batch[0]}\n')
        # print(f'states_batch[0] : {states_batch[0]}\n')
        # print(f'mixed_contexts_batch[0] : {mixed_contexts_batch[0]}\n')
        # print(f'fact_idxs_batch[0] : {fact_idxs_batch[0]}\n')
        # print(f'counter_idxs_batch[0] : {counter_idxs_batch[0]}\n')
        # print(f'attn_results_batch[0] : {attn_results_batch[0]}\n')
        # sys.exit(-1)



        idx = -1
        for j in range(batch_size1):
            for file_format in FILE_FORMATS:
                for k in range(CONTEXT_SIZE):
                    idx += 1

                    ext_n_fact = idx % CONTEXT_SIZE
                    ext_n_counter = CONTEXT_SIZE - 1 - ext_n_fact
                    ext_n_key = f'{ext_n_fact}\t{ext_n_counter}'

                    # 기본 key 저장 용도
                    add_str_num(percents, f'{file_format}\t{ext_n_key}', 0.0)
                    add_str_num(percents, f'ALL\t{ext_n_key}', 0.0)

                    if states_batch[idx]:
                        attn_result = attn_results_batch[idx]

                        fact_idxs = fact_idxs_batch[idx]
                        counter_idxs = counter_idxs_batch[idx]

                        fact_attn_percent_of_prompt = sum(attn_result['context_percents_of_prompt'].get(f'{context_idx}', 0.0) for context_idx in fact_idxs)
                        counter_attn_percent_of_prompt = sum(attn_result['context_percents_of_prompt'].get(f'{context_idx}', 0.0) for context_idx in counter_idxs)
                        add_str_num(percents, f'{file_format}_fact_of_prompt\t{ext_n_key}', fact_attn_percent_of_prompt)
                        add_str_num(percents, f'ALL_fact_of_prompt\t{ext_n_key}', fact_attn_percent_of_prompt)
                        add_str_num(percents, f'{file_format}_counter_of_prompt\t{ext_n_key}', counter_attn_percent_of_prompt)
                        add_str_num(percents, f'ALL_counter_of_prompt\t{ext_n_key}', counter_attn_percent_of_prompt)

                        fact_attn_percent_of_context_total = sum(attn_result['context_percents_of_context_total'].get(f'{context_idx}', 0.0) for context_idx in fact_idxs)
                        counter_attn_percent_of_context_total = sum(attn_result['context_percents_of_context_total'].get(f'{context_idx}', 0.0) for context_idx in counter_idxs)
                        add_str_num(percents, f'{file_format}_fact_of_context_total\t{ext_n_key}', fact_attn_percent_of_context_total)
                        add_str_num(percents, f'ALL_fact_of_context_total\t{ext_n_key}', fact_attn_percent_of_context_total)
                        add_str_num(percents, f'{file_format}_counter_of_context_total\t{ext_n_key}', counter_attn_percent_of_context_total)
                        add_str_num(percents, f'ALL_counter_of_context_total\t{ext_n_key}', counter_attn_percent_of_context_total)

        if (i+1) % 100 == 0:
            print(f'experiment_context_ratio() {(i+1)*batch_size1} complet.')
    print(f'experiment_context_ratio() {data_size} complet.\n')





    file_formats = FILE_FORMATS + ['ALL']
    for file_format in file_formats:
        for i in range(CONTEXT_SIZE):
            ext_n_key = f'{CONTEXT_SIZE-1-i}\t{i}'
            base_key = f'{file_format}\t{ext_n_key}'

            if base_key in percents.keys():
                sum_fact_attn_percent_of_prompt = percents[f'{file_format}_fact_of_prompt\t{ext_n_key}']
                sum_counter_attn_percent_of_prompt = percents[f'{file_format}_counter_of_prompt\t{ext_n_key}']
                sum_fact_attn_percent_of_context_total = percents[f'{file_format}_fact_of_context_total\t{ext_n_key}']
                sum_counter_attn_percent_of_context_total = percents[f'{file_format}_counter_of_context_total\t{ext_n_key}']

                if file_format == 'ALL':
                    size = data_size * len(FILE_FORMATS)
                else:
                    size = data_size

                print_str = f'{base_key}\t{size}\t{sum_fact_attn_percent_of_prompt / size}'
                print_str += f'\t{sum_counter_attn_percent_of_prompt / size}'
                print_str += f'\t{sum_fact_attn_percent_of_context_total / size}'
                print_str += f'\t{sum_counter_attn_percent_of_context_total / size}'
                print(print_str)
        print()

In [ ]:
# flash는 output_attentions 반환 안됨
model_utils.ATTN_IMP = 'eager'

model_names = ['Llama-3.2-3B', 'Llama-3.1-8B']
batch_sizes = [(1,2), (1,1)]

for model_name, (batch_size1, batch_size2) in zip(model_names, batch_sizes):
    if model_name.startswith(f'Llama'):
        model_name_or_path = f'meta-llama/{model_name}-Instruct'
    else:
        model_name_or_path = model_name
    
    model = model_utils.get_model(model_name_or_path, dtype, device=device, is_eval=True)

    # # 평가/추론 시에는 반드시 'left' 패딩
    tokenizer: PreTrainedTokenizerFast = tokenizer_utils.load_tokenizer(model_name_or_path, 'left')

    # model = None
    # tokenizer = None

    for zero_shot in ['fact', 'counter', 'other']:
        in_file_path = f'{in_dir}/{model_name}/bait_{model_name}_zero_shot_{zero_shot}_created_contexts.json'
        datas = json_utils.load_json(in_file_path)

        check_confirmation_bias(model, tokenizer, datas, batch_size1, batch_size2)

    del model
    del tokenizer
    common_utils.clear_gpu_memory()